# 📋 Dive.ai 팀 공유 문서
## 데이터셋 활용 방식 비교 계획
### 메타데이터 RAG vs 장르별 구조화 프롬프트

> **목적**: 두 방식 중 어떤 방식이 Dive.ai 시나리오 생성에 더 적합한지 비교 테스트 계획을 공유합니다.
> 벡터 DB 구축 전, 라벨링 데이터에서 직접 메타데이터를 필터링하는 방식으로 시뮬레이션합니다.

---

## 1. 왜 이 두 가지를 비교하는가?

### Few-shot는 왜 제외했나?

처음엔 세 가지 방식을 고려했어요.

| 방식 | 설명 |
|------|------|
| Few-shot | 실제 씬 텍스트 전문을 검색해서 프롬프트에 주입 |
| 메타데이터 RAG | 씬의 stage/motif 등 메타데이터만 검색해서 주입 |
| 장르별 구조화 프롬프트 | 장르별 통계 가이드만 제공, 스토리는 LLM에 전적으로 위임 |

**Few-shot을 제외한 이유**는 문화콘텐츠 데이터의 씬 텍스트가 캐릭터명이 C001, C002로 익명화되어 있고, 텍스트를 통째로 주입하면 LLM이 그 패턴에 끌려가서 오히려 창작 자유도가 떨어질 우려가 있기 때문입니다.

### 메타데이터 RAG vs 구조 설계만을 비교하는 이유

이 두 방식은 **데이터셋 의존도**와 **LLM 창작 자유도** 사이의 균형점이 달라요.

```
데이터셋 의존도  높음 ◀─────────────────────────▶ 낮음
                  텍스트 RAG  |  메타데이터 RAG  |  구조 설계만
LLM 자유도       낮음 ◀─────────────────────────▶ 높음
```

어느 쪽이 실제로 더 나은 시나리오를 만들어내는지 결과물로 직접 확인하는 것이 목표입니다.

---

## 2. 두 방식 상세 설명

### 🅐 메타데이터 RAG

유저가 소재를 입력하면 문화콘텐츠 라벨링 데이터에서 **유사한 장르·모티프의 씬 메타데이터**를 검색해서 LLM 프롬프트에 주입합니다. 씬 텍스트 전문이 아니라 구조적 정보(stage, motif, storyline 요약)만 가져옵니다.

**작동 흐름**
```
유저 입력: "배신당한 복수극" + 장르: 스릴러
       ↓
라벨링 DB에서 스릴러 장르 씬 검색
       ↓
유사도 높은 씬의 메타데이터 추출
  - stage: "Point of No Return"
  - unit_motif: "결단"
  - storyline: "주인공이 돌아올 수 없는 선택을 한다"
       ↓
LLM 프롬프트에 메타데이터만 주입
       ↓
시나리오 생성
```

**프롬프트 예시**
```
[유저 소재] 배신당한 복수극 (장르: 스릴러)

[유사 작품 씬 패턴 참고]
- [Point of No Return / 결단] 주인공이 돌아올 수 없는 선택을 한다
- [Climax / 역전] 모든 것을 잃은 순간, 판세가 뒤집힌다
- [Dark Night of Soul / 절망] 주인공이 혼자 모든 것을 감당해야 하는 밤
```

**기대 효과**: 유저 소재와 유사한 실제 흥행작의 서사 구조를 참고할 수 있어 장르 특성이 살아있는 시나리오 생성 가능

**우려 사항**: 검색된 씬 패턴이 비슷한 경우 시나리오가 비슷한 구조로 수렴할 수 있음

---

### 🅑 장르별 구조화 프롬프트 (통계 가이드)

씬 메타데이터를 검색하지 않고, 전체 데이터셋에서 미리 추출한 **장르별 통계 가이드**만 LLM에게 제공합니다. 스토리 생성은 LLM에 전적으로 위임합니다.

**작동 흐름**
```
유저 입력: "배신당한 복수극" + 장르: 스릴러
       ↓
사전에 추출해둔 스릴러 장르 통계 로드
  - 자주 등장하는 stage Top 5
  - 자주 쓰이는 motif Top 5
  - 주요 theme Top 3
       ↓
통계 가이드만 LLM 프롬프트에 주입
       ↓
시나리오 생성
```

**프롬프트 예시**
```
[유저 소재] 배신당한 복수극 (장르: 스릴러)

[스릴러 장르 흥행 패턴 가이드]
- 핵심 서사 단계: Inciting Incident, Point of No Return, Climax ...
- 자주 쓰이는 모티프: 배신, 복수, 역전, 추격, 비밀 폭로
- 주요 테마: 신뢰와 배신, 정의, 생존
```

**기대 효과**: LLM 창작 자유도가 높아 신선하고 독창적인 시나리오가 나올 가능성이 있음

**우려 사항**: 장르 특성이 덜 반영되거나 유저 소재와 동떨어진 방향으로 전개될 수 있음

---

## 3. 두 방식 비교 요약

| 항목 | 🅐 메타데이터 RAG | 🅑 구조 설계만 |
|------|-----------------|----------------|
| 데이터셋 활용 방식 | 유저 입력마다 유사 씬 메타데이터 검색 | 사전 추출 통계 가이드 고정 사용 |
| LLM 창작 자유도 | 상대적으로 낮음 | 높음 |
| 장르 특성 반영도 | 높음 (실제 씬 패턴 기반) | 보통 (통계 기반) |
| 시나리오 다양성 | 유저 소재에 따라 달라짐 | 항상 높음 |
| 구현 복잡도 | 높음 (벡터 DB 필요) | 낮음 (통계 파일 로드만) |
| 응답 속도 | 상대적으로 느림 (검색 과정 포함) | 빠름 |
| 데이터셋 의존도 | 중간 | 낮음 |

---

## 4. 비교 테스트 계획


### 테스트 진행 순서

```
1단계 — 전처리
  문화콘텐츠 라벨링 데이터에서 장르별 메타데이터 추출
  구조 설계용 통계 파일 생성 (genre_stage_stats, genre_motif_stats 등)

2단계 — 메타데이터 RAG 시뮬레이션 구현
  벡터 DB 구축 전, JSON 파일에서 직접 장르/모티프 필터링으로 시뮬레이션
  (실제 벡터 DB와 결과 품질 차이 거의 없음)

3단계 — 동일 소재로 두 방식 결과 생성
  같은 소재를 두 방식에 각각 입력하여 시나리오 생성

4단계 — 결과 분석 및 방식 결정
  장르별 패턴 분석 후 최종 방식 결정
  필요 시 하이브리드 전략 검토
```

### 평가 기준

| 평가 항목 | 설명 |
|-----------|------|
| 🎯 장르 느낌 | 선택한 장르의 분위기가 시나리오에 살아있는가 |
| 🌿 창의성 | 뻔하지 않고 신선한 전개인가 |
| 🔀 분기점 자연스러움 | 선택지가 서사 흐름에서 자연스럽게 나오는가 |
| 📖 완성도 | 기승전결이 뚜렷하고 읽기 좋은가 |
---

## 5. 예상 시나리오 및 다음 단계

### 예상 결과 시나리오

**시나리오 A — 메타데이터 RAG가 전반적으로 우세한 경우**
→ 벡터 DB 구축 진행, 전 장르에 메타데이터 RAG 적용

**시나리오 B — 구조 설계만이 전반적으로 우세한 경우**
→ 벡터 DB 구축 불필요, 구조 설계만으로 진행. 개발 복잡도와 비용 모두 절감

**시나리오 C — 장르별로 결과가 엇갈리는 경우**
→ 하이브리드 전략 검토. 예: 스릴러는 메타데이터 RAG, 로맨스·판타지는 구조 설계만

### 다음 단계

```
테스트 결과에 따라 결정
       ↓
메타데이터 RAG 채택 시     →  전처리 → 벡터 DB 구축 → RAG 파이프라인 구현
구조 설계만 채택 시        →  전처리 → 통계 파일 생성 → 프롬프트 설계
하이브리드 전략 채택 시    →  위 두 가지를 장르별로 분기 적용
```